# Train a Model for Detecting Solar Panels

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/train_solar_panel_detection.ipynb)

## Install package
To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [6]:
%pip install geoai-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 667.9/667.9 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.1/688.1 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.3/859.3 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6

## Import libraries

In [7]:
import geoai

## Download sample data

In [8]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:

# Please ensure your files 'solar_panels_davis_ca.tif', 'solar_panels_davis_ca.geojson',
# and 'solar_panels_test_davis_ca.tif' are located in a folder named 'geoai_data'
# within your Google Drive's 'My Drive' for these paths to be correct.
train_raster_url = "/content/drive/MyDrive/training/training/Orthomosaic/AIT_MAP.tif"
train_vector_url = "/content/drive/MyDrive/training/training/panels.geojson"
test_raster_url="/content/drive/MyDrive/test/test/Orthomosaic"

In [10]:
train_raster_path = train_raster_url
train_vector_path = train_vector_url
test_raster_path = test_raster_url

## Visualize sample data

## Create training data

In [17]:
out_folder = "output"
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_vector_path,
    tile_size=512,
    stride=384,
    buffer_radius=0,
)


Raster info for /content/drive/MyDrive/training/training/Orthomosaic/AIT_MAP.tif:
  CRS: EPSG:32647
  Dimensions: 110590 x 59904
  Resolution: (0.01635509999999997, 0.01635510000000083)
  Bands: 3
  Bounds: BoundingBox(left=672838.7392584, bottom=1556679.8572488, right=674647.4497674, top=1557659.5931592002)
Loaded 460 features from /content/drive/MyDrive/training/training/panels.geojson
Vector CRS: EPSG:32647


Generated: 44928, With features: 538: 100%|██████████| 44928/44928 [28:52<00:00, 25.94it/s]



------- Export Summary -------
Total tiles exported: 44928
Tiles with features: 538 (1.2%)
Average feature pixels per tile: 84141.7
Output saved to: output

------- Georeference Verification -------


In [14]:
detector = geoai.SolarPanelDetector()

Model path not specified, downloading from Hugging Face...


solar_panel_detection.pth:   0%|          | 0.00/176M [00:00<?, ?B/s]

Model downloaded to: /root/.cache/huggingface/hub/models--giswqs--geoai/snapshots/089548329c81f128fa12576663e7abdedb5cfa0e/solar_panel_detection.pth
Model loaded successfully


In [2]:
output_path = "solar_panel_masks.tif"

In [1]:
masks_path = detector.generate_masks(
    raster_path=train_raster_path,          # same ortho
    output_path="solar_panel_masks.tif",          # NEW folder
    confidence_threshold=0.55,
    mask_threshold=0.6,
    min_object_area=250,
    overlap=0.25,
    chip_size=(512, 512),                   # match tile size
    batch_size=4,
    verbose=False,
)

NameError: name 'detector' is not defined

## Train object detection model

In [ ]:
geoai.train_MaskRCNN_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/models",
    num_channels=3,
    pretrained=True,
    batch_size=4,
    num_epochs=100,
    learning_rate=0.005,
    val_split=0.2,
)

## Run inference

In [ ]:
masks_path = "solar_panels_prediction.tif"
model_path = f"{out_folder}/models/best_model.pth"

In [ ]:
geoai.object_detection(
    test_raster_path,
    masks_path,
    model_path,
    window_size=512,
    overlap=256,
    confidence_threshold=0.5,
    batch_size=4,
    num_channels=3,
)

## Vectorize masks

In [ ]:
output_path = "solar_panels_prediction.geojson"
gdf = geoai.orthogonalize(masks_path, output_path, epsilon=2)

## Visualize results

In [ ]:
geoai.view_vector_interactive(output_path, tiles=test_raster_url)

In [ ]:
geoai.create_split_map(
    left_layer=output_path,
    right_layer=test_raster_url,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_url,
)

![image](https://github.com/user-attachments/assets/69d20aec-991f-4681-b31e-fabcd7e21b91)